# Kaggle Model 2026
Replicates the feature engineering and model architecture from `final-solution-ncaa-2025.ipynb`.
Trains three models and saves them to `model/`:
- `kaggle_proba.joblib` — leave-one-season-out XGBoost boosters + spline calibration for win probability
- `kaggle_spread.joblib` — XGBRegressor for point spread (PointDiff)
- `kaggle_total.joblib` — XGBRegressor for total score (T1_Score + T2_Score)

In [ ]:
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import statsmodels.api as sm
import tqdm
import xgboost as xgb
from scipy.interpolate import UnivariateSpline
from sklearn.metrics import brier_score_loss, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_column", 999)

## Load data

In [ ]:
data_dir = "data_2026"

M_regular_results = pd.read_csv(f"{data_dir}/MRegularSeasonDetailedResults.csv")
M_tourney_results = pd.read_csv(f"{data_dir}/MNCAATourneyDetailedResults.csv")
M_seeds = pd.read_csv(f"{data_dir}/MNCAATourneySeeds.csv")

W_regular_results = pd.read_csv(f"{data_dir}/WRegularSeasonDetailedResults.csv")
W_tourney_results = pd.read_csv(f"{data_dir}/WNCAATourneyDetailedResults.csv")
W_seeds = pd.read_csv(f"{data_dir}/WNCAATourneySeeds.csv")

# Join men's and women's data
regular_results = pd.concat([M_regular_results, W_regular_results])
tourney_results = pd.concat([M_tourney_results, W_tourney_results])
seeds = pd.concat([M_seeds, W_seeds])

season_cutoff = 2003
regular_results = regular_results.loc[regular_results["Season"] >= season_cutoff]
tourney_results = tourney_results.loc[tourney_results["Season"] >= season_cutoff]
seeds = seeds.loc[seeds["Season"] >= season_cutoff]

## Prepare data
Doubles the dataset by swapping team positions. Adjusts stats for overtime periods.
Exact replication of `prepare_data` from final-solution-ncaa-2025.

In [ ]:
def prepare_data(df):
    df = df[["Season", "DayNum", "LTeamID", "LScore", "WTeamID", "WScore", "NumOT",
             "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
             "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF"]]

    # Adjust all counting stats for overtime (normalise to 40 regulation minutes)
    adjot = (40 + 5 * df["NumOT"]) / 40
    adjcols = ["LScore", "WScore",
               "LFGM", "LFGA", "LFGM3", "LFGA3", "LFTM", "LFTA", "LOR", "LDR", "LAst", "LTO", "LStl", "LBlk", "LPF",
               "WFGM", "WFGA", "WFGM3", "WFGA3", "WFTM", "WFTA", "WOR", "WDR", "WAst", "WTO", "WStl", "WBlk", "WPF"]
    for col in adjcols:
        df[col] = df[col] / adjot

    dfswap = df.copy()
    df.columns = [x.replace("W", "T1_").replace("L", "T2_") for x in list(df.columns)]
    dfswap.columns = [x.replace("L", "T1_").replace("W", "T2_") for x in list(dfswap.columns)]
    output = pd.concat([df, dfswap]).reset_index(drop=True)
    output["PointDiff"] = output["T1_Score"] - output["T2_Score"]
    output["win"] = (output["PointDiff"] > 0) * 1
    output["men_women"] = (output["T1_TeamID"].apply(lambda t: str(t).startswith("1"))) * 1  # 1: men, 0: women
    return output


regular_data = prepare_data(regular_results)
tourney_data = prepare_data(tourney_results)
print(f"regular_data: {regular_data.shape}, tourney_data: {tourney_data.shape}")

## Easy features — seeds

In [ ]:
seeds["seed"] = seeds["Seed"].apply(lambda x: int(x[1:3]))

seeds_T1 = seeds[["Season", "TeamID", "seed"]].copy()
seeds_T2 = seeds[["Season", "TeamID", "seed"]].copy()
seeds_T1.columns = ["Season", "T1_TeamID", "T1_seed"]
seeds_T2.columns = ["Season", "T2_TeamID", "T2_seed"]

# Preserve T1_Score + T2_Score before stripping tourney_data down to core columns
tourney_data["Total"] = tourney_data["T1_Score"] + tourney_data["T2_Score"]

tourney_data = tourney_data[["Season", "T1_TeamID", "T2_TeamID", "PointDiff", "win", "men_women", "Total"]]
tourney_data = pd.merge(tourney_data, seeds_T1, on=["Season", "T1_TeamID"], how="left")
tourney_data = pd.merge(tourney_data, seeds_T2, on=["Season", "T2_TeamID"], how="left")
tourney_data["Seed_diff"] = tourney_data["T2_seed"] - tourney_data["T1_seed"]

tourney_data.head()

## Medium features — regular season averages

In [ ]:
boxcols = [
    "T1_Score", "T1_FGM", "T1_FGA", "T1_FGM3", "T1_FGA3", "T1_FTM", "T1_FTA",
    "T1_OR", "T1_DR", "T1_Ast", "T1_TO", "T1_Stl", "T1_Blk", "T1_PF",
    "T2_Score", "T2_FGM", "T2_FGA", "T2_FGM3", "T2_FGA3", "T2_FTM", "T2_FTA",
    "T2_OR", "T2_DR", "T2_Ast", "T2_TO", "T2_Stl", "T2_Blk", "T2_PF",
    "PointDiff",
]

ss = regular_data.groupby(["Season", "T1_TeamID"])[boxcols].agg("mean").reset_index()

ss_T1 = ss.copy()
ss_T1.columns = ["T1_avg_" + x.replace("T1_", "").replace("T2_", "opponent_") for x in list(ss_T1.columns)]
ss_T1 = ss_T1.rename({"T1_avg_Season": "Season", "T1_avg_TeamID": "T1_TeamID"}, axis=1)

ss_T2 = ss.copy()
ss_T2.columns = ["T2_avg_" + x.replace("T1_", "").replace("T2_", "opponent_") for x in list(ss_T2.columns)]
ss_T2 = ss_T2.rename({"T2_avg_Season": "Season", "T2_avg_TeamID": "T2_TeamID"}, axis=1)

tourney_data = pd.merge(tourney_data, ss_T1, on=["Season", "T1_TeamID"], how="left")
tourney_data = pd.merge(tourney_data, ss_T2, on=["Season", "T2_TeamID"], how="left")
print(tourney_data.shape)

## Hard features — ELO ratings

In [ ]:
def expected_result(elo_a, elo_b):
    return 1.0 / (1 + 10 ** ((elo_b - elo_a) / elo_width))


def update_elo(winner_elo, loser_elo):
    expected_win = expected_result(winner_elo, loser_elo)
    change_in_elo = k_factor * (1 - expected_win)
    winner_elo += change_in_elo
    loser_elo -= change_in_elo
    return winner_elo, loser_elo


base_elo = 1000
elo_width = 400
k_factor = 100

elos = []
for season in sorted(set(seeds["Season"])):
    ss_elo = regular_data.loc[regular_data["Season"] == season]
    ss_elo = ss_elo.loc[ss_elo["win"] == 1].reset_index(drop=True)
    teams = set(ss_elo["T1_TeamID"]) | set(ss_elo["T2_TeamID"])
    elo = dict(zip(teams, [base_elo] * len(teams)))
    for i in range(ss_elo.shape[0]):
        w_team, l_team = ss_elo.loc[i, "T1_TeamID"], ss_elo.loc[i, "T2_TeamID"]
        w_elo_new, l_elo_new = update_elo(elo[w_team], elo[l_team])
        elo[w_team] = w_elo_new
        elo[l_team] = l_elo_new
    elo_df = pd.DataFrame.from_dict(elo, orient="index").reset_index()
    elo_df = elo_df.rename({"index": "TeamID", 0: "elo"}, axis=1)
    elo_df["Season"] = season
    elos.append(elo_df)

elos = pd.concat(elos)
elos_T1 = elos.copy().rename({"TeamID": "T1_TeamID", "elo": "T1_elo"}, axis=1)
elos_T2 = elos.copy().rename({"TeamID": "T2_TeamID", "elo": "T2_elo"}, axis=1)

tourney_data = pd.merge(tourney_data, elos_T1, on=["Season", "T1_TeamID"], how="left")
tourney_data = pd.merge(tourney_data, elos_T2, on=["Season", "T2_TeamID"], how="left")
tourney_data["elo_diff"] = tourney_data["T1_elo"] - tourney_data["T2_elo"]
print(tourney_data.shape)

## Hardest features — GLM team quality
Fits a Gaussian GLM per season/gender on regular season results to estimate a latent team strength.
Only includes tourney teams (+ teams that beat a tourney team).

In [ ]:
regular_data["ST1"] = regular_data.apply(lambda t: str(int(t["Season"])) + "/" + str(int(t["T1_TeamID"])), axis=1)
regular_data["ST2"] = regular_data.apply(lambda t: str(int(t["Season"])) + "/" + str(int(t["T2_TeamID"])), axis=1)
seeds_T1["ST1"] = seeds_T1.apply(lambda t: str(int(t["Season"])) + "/" + str(int(t["T1_TeamID"])), axis=1)
seeds_T2["ST2"] = seeds_T2.apply(lambda t: str(int(t["Season"])) + "/" + str(int(t["T2_TeamID"])), axis=1)

# Tourney teams, plus non-tourney teams that beat a tourney team at least once
st = set(seeds_T1["ST1"]) | set(seeds_T2["ST2"])
st = st | set(regular_data.loc[
    (regular_data["T1_Score"] > regular_data["T2_Score"]) & (regular_data["ST2"].isin(st)),
    "ST1"
])

dt = regular_data.loc[regular_data["ST1"].isin(st) | regular_data["ST2"].isin(st)].copy()
dt["T1_TeamID"] = dt["T1_TeamID"].astype(str)
dt["T2_TeamID"] = dt["T2_TeamID"].astype(str)
dt.loc[~dt["ST1"].isin(st), "T1_TeamID"] = "0000"
dt.loc[~dt["ST2"].isin(st), "T2_TeamID"] = "0000"


def team_quality(season, men_women):
    formula = "PointDiff~-1+T1_TeamID+T2_TeamID"
    glm = sm.GLM.from_formula(
        formula=formula,
        data=dt.loc[(dt["Season"] == season) & (dt["men_women"] == men_women), :],
        family=sm.families.Gaussian(),
    ).fit()
    quality = pd.DataFrame(glm.params).reset_index()
    quality.columns = ["TeamID", "quality"]
    quality["Season"] = season
    quality = quality.loc[quality.TeamID.str.contains("T1_")].reset_index(drop=True)
    quality["TeamID"] = quality["TeamID"].apply(lambda x: x[10:14]).astype(int)
    return quality


glm_quality = []
seasons = sorted(set(seeds["Season"]))
for s in tqdm.tqdm(seasons, unit="season"):
    if s >= 2010:  # women's data starts 2010
        glm_quality.append(team_quality(s, 0))
    if s >= 2003:  # men's data starts 2003
        glm_quality.append(team_quality(s, 1))

glm_quality = pd.concat(glm_quality).reset_index(drop=True)
glm_quality_T1 = glm_quality.copy()
glm_quality_T2 = glm_quality.copy()
glm_quality_T1.columns = ["T1_TeamID", "T1_quality", "Season"]
glm_quality_T2.columns = ["T2_TeamID", "T2_quality", "Season"]

tourney_data = pd.merge(tourney_data, glm_quality_T1, on=["Season", "T1_TeamID"], how="left")
tourney_data = pd.merge(tourney_data, glm_quality_T2, on=["Season", "T2_TeamID"], how="left")
tourney_data["diff_quality"] = tourney_data["T1_quality"] - tourney_data["T2_quality"]
print(tourney_data.shape)

## Feature list

In [ ]:
features = [
    ### EASY FEATURES ###
    "men_women",
    "T1_seed",
    "T2_seed",
    "Seed_diff",
    ### MEDIUM FEATURES ###
    "T1_avg_Score",
    "T1_avg_FGA",
    "T1_avg_OR",
    "T1_avg_DR",
    "T1_avg_Blk",
    "T1_avg_PF",
    "T1_avg_opponent_FGA",
    "T1_avg_opponent_Blk",
    "T1_avg_opponent_PF",
    "T1_avg_PointDiff",
    "T2_avg_Score",
    "T2_avg_FGA",
    "T2_avg_OR",
    "T2_avg_DR",
    "T2_avg_Blk",
    "T2_avg_PF",
    "T2_avg_opponent_FGA",
    "T2_avg_opponent_Blk",
    "T2_avg_opponent_PF",
    "T2_avg_PointDiff",
    ### HARD FEATURES ###
    "T1_elo",
    "T2_elo",
    "elo_diff",
    ### HARDEST FEATURES ###
    "T1_quality",
    "T2_quality",
]

print(f"Number of features: {len(features)}")
missing = [f for f in features if f not in tourney_data.columns]
if missing:
    print(f"WARNING — missing columns: {missing}")
else:
    print("All features present.")

## Train win probability model
Leave-one-season-out XGBoost trained on PointDiff. Predictions converted to probabilities via a fitted spline.

In [ ]:
param = {
    "objective": "reg:squarederror",
    "booster": "gbtree",
    "eta": 0.0093,
    "subsample": 0.6,
    "colsample_bynode": 0.8,
    "num_parallel_tree": 2,
    "min_child_weight": 4,
    "max_depth": 4,
    "tree_method": "hist",
    "grow_policy": "lossguide",
    "max_bin": 38,
}
num_rounds = 704

models = {}
oof_mae = []
oof_preds = []
oof_targets = []
oof_ss = []

for oof_season in sorted(set(tourney_data.Season)):
    x_train = tourney_data.loc[tourney_data["Season"] != oof_season, features].values
    y_train = tourney_data.loc[tourney_data["Season"] != oof_season, "PointDiff"].values
    x_val   = tourney_data.loc[tourney_data["Season"] == oof_season, features].values
    y_val   = tourney_data.loc[tourney_data["Season"] == oof_season, "PointDiff"].values
    s_val   = tourney_data.loc[tourney_data["Season"] == oof_season, "Season"].values

    dtrain = xgb.DMatrix(x_train, label=y_train)
    dval   = xgb.DMatrix(x_val,   label=y_val)
    models[oof_season] = xgb.train(params=param, dtrain=dtrain, num_boost_round=num_rounds)

    preds = models[oof_season].predict(dval)
    mae = mean_absolute_error(y_val, preds)
    print(f"  season {oof_season}  MAE: {mae:.3f}")
    oof_mae.append(mae)
    oof_preds   += list(preds)
    oof_targets += list(y_val)
    oof_ss      += list(s_val)

print(f"\nAverage OOF MAE: {np.mean(oof_mae):.3f}")

## Spline calibration — convert predicted margin to win probability

In [ ]:
t = 25  # clip threshold for spline

dat = sorted(zip(oof_preds, np.array(oof_targets) > 0), key=lambda x: x[0])
pred_sorted, label_sorted = zip(*dat)
spline_model = UnivariateSpline(np.clip(pred_sorted, -t, t), label_sorted, k=5)

spline_fit = np.clip(spline_model(np.clip(oof_preds, -t, t)), 0.01, 0.99)
print(f"OOF Brier score: {brier_score_loss(np.array(oof_targets) > 0, spline_fit):.5f}")

# Per-season Brier scores
oof_df = pd.DataFrame({"Season": oof_ss, "pred": spline_fit, "label": (np.array(oof_targets) > 0) * 1})
for s in sorted(oof_df.Season.unique()):
    sub = oof_df[oof_df.Season == s]
    print(f"  {s}  Brier: {brier_score_loss(sub.label, sub.pred):.5f}")

## Win prediction accuracy

In [ ]:
from sklearn.metrics import accuracy_score

oof_win_pred   = (np.array(oof_preds) > 0).astype(int)
oof_win_actual = (np.array(oof_targets) > 0).astype(int)

overall_acc = accuracy_score(oof_win_actual, oof_win_pred)

rows = []
for s in sorted(set(oof_ss)):
    mask   = np.array(oof_ss) == s
    acc    = accuracy_score(oof_win_actual[mask], oof_win_pred[mask])
    gender = "M" if tourney_data.loc[tourney_data.Season == s, "men_women"].iloc[0] == 1 else "W"
    rows.append({"Season": s, "Gender": gender, "Accuracy": round(acc, 4), "Games": int(mask.sum())})

acc_df = pd.DataFrame(rows)
print(f"Overall OOF accuracy: {overall_acc:.4f}  ({overall_acc*100:.2f}%)")
print(f"Men's   OOF accuracy: {acc_df[acc_df.Gender == 'M']['Accuracy'].mean():.4f}")
print(f"Women's OOF accuracy: {acc_df[acc_df.Gender == 'W']['Accuracy'].mean():.4f}")
print()
acc_df

## Train spread and total models
Single XGBRegressor trained on all historical tourney data with the same features and hyperparameters.

In [ ]:
xgb_params = dict(
    objective="reg:squarederror",
    n_estimators=num_rounds,
    learning_rate=0.0093,
    subsample=0.6,
    colsample_bynode=0.8,
    num_parallel_tree=2,
    min_child_weight=4,
    max_depth=4,
    tree_method="hist",
    grow_policy="lossguide",
)

X_all = tourney_data[features]
y_spread = tourney_data["PointDiff"]
y_total  = tourney_data["Total"]

spread_model = XGBRegressor(**xgb_params)
spread_model.fit(X_all, y_spread)

total_model = XGBRegressor(**xgb_params)
total_model.fit(X_all, y_total)

# Evaluate on 2024-2025 test seasons
test_mask = tourney_data["Season"].isin([2024, 2025])
test_data = tourney_data[test_mask]
X_test = test_data[features]

spread_mae = mean_absolute_error(test_data["PointDiff"], spread_model.predict(X_test))
total_mae  = mean_absolute_error(test_data["Total"],     total_model.predict(X_test))
print(f"Spread MAE (2024-2025): {spread_mae:.2f} pts")
print(f"Total  MAE (2024-2025): {total_mae:.2f} pts")

## Save models

In [ ]:
os.makedirs("model", exist_ok=True)

# kaggle_proba: all season boosters + spline + feature list
joblib.dump(
    {"models": models, "spline": spline_model, "t": t, "features": features},
    "model/kaggle_proba.joblib",
)

joblib.dump(spread_model, "model/kaggle_spread.joblib")
joblib.dump(total_model,  "model/kaggle_total.joblib")

print("Saved:")
print("  model/kaggle_proba.joblib")
print("  model/kaggle_spread.joblib")
print("  model/kaggle_total.joblib")

## Generate Kaggle submission
Loads `SampleSubmissionStage2.csv`, joins all features, averages predictions across season models, applies spline calibration.

In [ ]:
X = pd.read_csv(f"{data_dir}/SampleSubmissionStage2.csv")

X["Season"]    = X["ID"].apply(lambda t: int(t.split("_")[0]))
X["T1_TeamID"] = X["ID"].apply(lambda t: int(t.split("_")[1]))
X["T2_TeamID"] = X["ID"].apply(lambda t: int(t.split("_")[2]))
# men_women consistent with training encoding: 1=men (TeamID starts with '1'), 0=women
X["men_women"] = X["T1_TeamID"].apply(lambda t: 1 if str(t)[0] == "1" else 0)

X = pd.merge(X, ss_T1,          on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, ss_T2,          on=["Season", "T2_TeamID"], how="left")
X = pd.merge(X, seeds_T1,       on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, seeds_T2,       on=["Season", "T2_TeamID"], how="left")
X = pd.merge(X, glm_quality_T1, on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, glm_quality_T2, on=["Season", "T2_TeamID"], how="left")
X = pd.merge(X, elos_T1,        on=["Season", "T1_TeamID"], how="left")
X = pd.merge(X, elos_T2,        on=["Season", "T2_TeamID"], how="left")
X["Seed_diff"]    = X["T2_seed"]    - X["T1_seed"]
X["elo_diff"]     = X["T1_elo"]     - X["T2_elo"]
X["diff_quality"] = X["T1_quality"] - X["T2_quality"]

# Fill NaN feature values with column medians from training data
for col in features:
    if X[col].isna().any():
        X[col] = X[col].fillna(tourney_data[col].median())

print(f"Submission rows: {len(X)}")

In [ ]:
# Average predictions across all season models then apply spline calibration
dtest = xgb.DMatrix(X[features].values)

all_margin_preds = np.array([
    models[oof_season].predict(dtest)
    for oof_season in sorted(models.keys())
])
avg_margin = all_margin_preds.mean(axis=0)

X["Pred"] = np.clip(spline_model(np.clip(avg_margin, -t, t)), 0.01, 0.99)
X["Pred"] = X["Pred"].round(6)

X[["ID", "Pred"]].to_csv("submission_2026_kaggle.csv", index=False)
print("Saved submission_2026_kaggle.csv")
X[["ID", "Pred"]].describe()